In [1]:
import os
os.chdir('/home/asudupe/Latxa-Omni/')

In [ ]:
import torch
import torchaudio
from omni_speech.constants import SPEECH_TOKEN_INDEX, DEFAULT_SPEECH_TOKEN
from omni_speech.conversation import conv_templates, SeparatorStyle
from omni_speech.model.builder import load_pretrained_model
from omni_speech.datasets.preprocess import tokenizer_speech_token
from torch.utils.data import Dataset, DataLoader
import whisper
from datasets import load_dataset, load_from_disk
import numpy as np
from IPython.display import Audio
from scipy.io.wavfile import write
from torchaudio.transforms import Resample
from speechbrain.inference.vocoders import UnitHIFIGAN
from transformers import Wav2Vec2Processor, AutoTokenizer

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def ctc_postprocess(tokens, blank):
    _toks = tokens.squeeze(0).tolist()
    deduplicated_toks = [v for i, v in enumerate(_toks) if i == 0 or v != _toks[i - 1]]
    hyp = [v for v in deduplicated_toks if v != blank] #官方493 222
    hyp = " ".join(list(map(str, hyp))) #1918 547
    return hyp

In [ ]:
dataset = load_from_disk('/scratch/asudupe/datasets/VoiceAssistant-400K_eu/')

In [ ]:
speech, sr = torchaudio.load(os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['train'][10]['question_audio']))
Audio(data=np.array(speech), rate=sr)

In [ ]:
dataset['train'][10]['answer']

In [ ]:
write(filename='example.wav', data=np.array(dataset['train'][4]['question_audio'], dtype=np.float32), rate=22050)

In [ ]:
speech_file = "omni_speech/serve/examples/helpful_base_1.wav"
speech = whisper.load_audio(speech_file)

Audio(data=np.array(speech), rate=16000)


In [ ]:
# model_path = 'saves/13834/checkpoint-24000'
# model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/stage1/best/checkpoint-20976"
model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/stage1/3931299/checkpoint-13984"
# model_path = "/hitz_data/asudupe/models/Latxa-Llama-3.1-8B-Instruct"
# model_path = "Llama-3.1-8B-Omni"
model_base = None
is_lora = False
s2s = False
mel_size = 128
conv_mode = 'llama_3'

In [ ]:
tokenizer, model, context_len = load_pretrained_model(model_path, model_base, is_lora=is_lora, s2s=s2s)


In [ ]:
hifigan = UnitHIFIGAN.from_hparams(source="/scratch/asudupe/models/hifigan/sonora_2/", run_opts={"device":'cuda'})

In [ ]:
qs = "<speech>\nPlease directly answer the questions in the user's speech."
speech_file = 'audioak/Recording 9.mp3'
# speech_file = os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['test'][1016]['question_audio'])
speech_loaded = whisper.load_audio(speech_file)
# audio = dataset['train'][20]['question_audio']
# speech = torch.tensor(audio, dtype=torch.float32)
# speech = Resample(orig_freq=22050, new_freq=16000)(speech)

conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

speech = whisper.pad_or_trim(speech_loaded)
speech = whisper.log_mel_spectrogram(speech, n_mels=mel_size).permute(1, 0)
# speech = hubert_tokenizer(speech_loaded, sampling_rate=16000, return_tensors="pt", padding=True)['input_values'].permute(1, 0)

input_ids = tokenizer_speech_token(prompt, tokenizer, return_tensors='pt')
speech_length = torch.LongTensor([speech.shape[0]])

input_ids = input_ids.to(device='cuda', non_blocking=True)
speech_tensor = speech.to(dtype=torch.float16, device='cuda', non_blocking=True)
speech_length = speech_length.to(device='cuda', non_blocking=True)

input_ids = input_ids.unsqueeze(0)
speech_tensors = speech_tensor.unsqueeze(0)
speech_lengths = speech_length.unsqueeze(0)

# input_ids = torch.stack((input_ids), dim=0)
# speech_tensors = torch.stack((speech_tensor), dim=0)
# speech_lengths = torch.stack((speech_length), dim=0)

#torch.Size([1, 62]),torch.Size([1, 3000, 128]) #tensor([[3000]])
# Audio(speech_loaded, rate=16000)

In [ ]:
input_ids.shape, speech_tensors.shape, speech_lengths

In [ ]:
temperature = 0
top_p = None
num_beams = 1
max_new_tokens = 512

with torch.inference_mode():
    outputs = model.generate(
        input_ids,
        speech=speech_tensors,
        speech_lengths=speech_lengths,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        pad_token_id=128004,
        streaming_unit_gen=True,
 
    )
# output_ids = outputs
output_ids, output_units = outputs

print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip())
output_units = ctc_postprocess(output_units, blank=model.config.unit_vocab_size)
output_units = torch.tensor([int(x) for x in output_units.split()], dtype=torch.long)
answer = hifigan.decode_unit(output_units.unsqueeze(-1), torch.tensor(np.load('/scratch/asudupe/models/hifigan/sonora_2/alex.npy')))
Audio(answer.cpu(), rate=16000)

In [ ]:
torchaudio.save("audioak/erantzuna.wav", answer.cpu(), sample_rate=16000)